# Padronização, distância e K-means

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_09/01_padronizacao_distancia_e_kmeans.ipynb)

Distância traduz “semelhança” em regra geométrica. Na distância euclidiana,
diferenças grandes pesam quadraticamente. Padronizar coloca cada variável em média
zero e desvio-padrão um; isso não a torna neutra, apenas muda a pergunta.

![A mesma nuvem produz vizinhanças diferentes antes e depois da padronização.](imagens/01_escala_distancia.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_09'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('sklearn', 'scikit-learn>=1.5,<2')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
dados=pd.read_csv("dados/instituicoes_culturais.csv")
variaveis=["itens_acervo","eventos_ano","publico_ano","orcamento_anual"]
X=dados[variaveis]
Xz=StandardScaler().fit_transform(X)
sem_escala=KMeans(n_clusters=3,random_state=42,n_init=20).fit_predict(X)
com_escala=KMeans(n_clusters=3,random_state=42,n_init=20).fit_predict(Xz)
comparacao=pd.crosstab(sem_escala,com_escala,rownames=["sem padronizar"],colnames=["padronizado"])
display(comparacao)

## 2. Escolher $k$ sem procurar um número mágico

K-means minimiza distâncias quadráticas aos centroides e pressupõe grupos
aproximadamente compactos. Inércia sempre cai quando $k$ cresce. Silhueta compara
coesão e separação entre -1 e 1, mas não decide relevância humanística.

In [ ]:
avaliacoes=[]
for k in range(2,7):
    rotulos=KMeans(n_clusters=k,random_state=42,n_init=20).fit_predict(Xz)
    avaliacoes.append({"k":k,"inercia":KMeans(n_clusters=k,random_state=42,n_init=20).fit(Xz).inertia_,"silhueta":silhouette_score(Xz,rotulos)})
tabela_k=pd.DataFrame(avaliacoes)
display(tabela_k.round(3))
fig,ax=plt.subplots(1,2,figsize=(10,4)); ax[0].plot(tabela_k.k,tabela_k.inercia,marker="o"); ax[0].set(title="Inércia",xlabel="k"); ax[1].plot(tabela_k.k,tabela_k.silhueta,marker="o"); ax[1].set(title="Silhueta",xlabel="k",ylim=(-1,1)); plt.tight_layout(); plt.show()
modelo_k=KMeans(n_clusters=3,random_state=42,n_init=20)
dados["cluster_kmeans"]=modelo_k.fit_predict(Xz)

### U09-A02 — Atividade integrada — escala e número de grupos

Compare rótulos com e sem padronização, examine $k=2$ a $6$ e escolha duas soluções
plausíveis. Para cada uma, diga o que a métrica favorece e que questão substantiva
ainda falta responder. **Dinâmica (30 minutos):** individual na execução e dupla na
decisão; não escolha apenas o maior valor.

**Resposta:** escreva aqui.

## Passagem

Uma partição não basta. Precisamos caracterizar seus casos, compará-la a outro
algoritmo e verificar se pequenas mudanças alteram a conclusão.